In [3]:
import pandas as pd

df = pd.read_csv('../data/raw/dataset-google.csv')

print(df.shape)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())



(1272, 11)
year                                 float64
sector                                   str
subsector                                str
industry_group                           str
industry                                 str
total_employed_in_thousands          float64
percent_women                        float64
percent_white                        float64
percent_black_or_african_american    float64
percent_asian                        float64
percent_hispanic_or_latino           float64
dtype: object
     year                         sector subsector industry_group industry  \
0  2022.0  Education and health services       NaN            NaN      NaN   
1  2021.0  Education and health services       NaN            NaN      NaN   
2  2023.0                  Manufacturing       NaN            NaN      NaN   
3  2021.0   Transportation and utilities       NaN            NaN      NaN   
4  2021.0       Total, 16 years and over       NaN            NaN      NaN   

   total_em

In [4]:
# understand the hierarchy
print("Years:", df['year'].unique())
print("\nTop sectors:\n", df['sector'].value_counts())
print("\nNull pattern:")
print(df[['subsector','industry_group','industry']].isnull().sum())

# check that outlier
print("\nHigh asian percent rows:")
print(df[df['percent_asian'] > 0.5][['sector','industry','percent_asian']])

Years: [2022. 2021. 2023. 2020.   nan]

Top sectors:
 sector
Manufacturing                                    387
Wholesale and retail trade                       236
Education and health services                    104
Professional and business services                92
Transportation and utilities                      84
Other services                                    84
Financial activities                              68
Leisure and hospitality                           64
Information                                       52
Public administration                             36
Mining, quarrying, and oil and gas extraction     28
Agriculture, forestry, fishing, and hunting       28
Total, 16 years and over                           4
Construction                                       4
Name: count, dtype: int64

Null pattern:
subsector          57
industry_group    257
industry          765
dtype: int64

High asian percent rows:
                                            sector 

In [6]:
df = df.dropna(subset=['year'])

df['year'] = df['year'].astype(int)

df = df[df['percent_asian'] <= 1.0]

demo_columns = ['percent_asian', 'percent_women','percent_black_or_african_american', 'percent_asian',
'percent_hispanic_or_latino']

print("Shape after drops:", df.shape)
print("\nRemaining nulls:")
print(df[demo_columns].isnull().sum())


print("\nYear distribution:")
print(df['year'].value_counts().sort_index())


Shape after drops: (1131, 11)

Remaining nulls:
percent_asian                        0
percent_women                        0
percent_black_or_african_american    0
percent_asian                        0
percent_hispanic_or_latino           0
dtype: int64

Year distribution:
year
2020    281
2021    284
2022    284
2023    282
Name: count, dtype: int64


In [8]:
# check which rows had null demographics BEFORE our drops
# reload fresh to compare
df_original = pd.read_csv("../data/raw/dataset-google.csv")

# rows with null demographics
null_demo_rows = df_original[df_original['percent_women'].isnull()]
print("Rows with null demographics:")
print(null_demo_rows[['year','sector','percent_asian','percent_women']].head(20))
print("\nShape of null rows:", null_demo_rows.shape)

Rows with null demographics:
       year                                         sector  percent_asian  \
18      NaN                                            NaN            NaN   
105  2020.0                        Leisure and hospitality            NaN   
106  2021.0                        Leisure and hospitality            NaN   
108  2023.0                        Leisure and hospitality            NaN   
137  2023.0  Mining, quarrying, and oil and gas extraction            NaN   
138  2022.0  Mining, quarrying, and oil and gas extraction            NaN   
140  2021.0  Mining, quarrying, and oil and gas extraction            NaN   
198  2022.0                                  Manufacturing            NaN   
215  2020.0                                  Manufacturing            NaN   
241  2020.0                                  Manufacturing            NaN   
245  2021.0                                  Manufacturing            NaN   
247  2022.0                                  Ma

In [13]:
df.info()
df = df[df['sector'] != 'Total, 16 years and over']

<class 'pandas.DataFrame'>
Index: 1131 entries, 0 to 1271
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   year                               1131 non-null   int64  
 1   sector                             1131 non-null   str    
 2   subsector                          1076 non-null   str    
 3   industry_group                     898 non-null    str    
 4   industry                           426 non-null    str    
 5   total_employed_in_thousands        1131 non-null   float64
 6   percent_women                      1131 non-null   float64
 7   percent_white                      1131 non-null   float64
 8   percent_black_or_african_american  1131 non-null   float64
 9   percent_asian                      1131 non-null   float64
 10  percent_hispanic_or_latino         1131 non-null   float64
dtypes: float64(6), int64(1), str(4)
memory usage: 106.0 KB


In [14]:
top10 = (df.groupby('sector')
         .agg({
             'total_employed_in_thousands': 'sum',
             'percent_women': 'mean',
             'year': 'count'                    # how many years/data points
         })
         .round(2)
         .rename(columns={'year': 'data_points'})
         .sort_values('total_employed_in_thousands', ascending=False)
         .head(10))

print(top10)

                                    total_employed_in_thousands  \
sector                                                            
Education and health services                          507565.0   
Professional and business services                     262633.0   
Manufacturing                                          236481.0   
Wholesale and retail trade                             232704.0   
Leisure and hospitality                                196004.0   
Financial activities                                   154339.0   
Transportation and utilities                           113503.0   
Other services                                         110972.0   
Public administration                                   61139.0   
Construction                                            33847.0   

                                    percent_women  data_points  
sector                                                          
Education and health services                0.74          104  


In [15]:
df['percent_women'].mean()

np.float64(0.4205146406388643)